In [ ]:
import os
import glob
import subprocess
import random
from moviepy import VideoFileClip

offsets = {"ad00": 106404, "ad01": 112595, "ad02": 72247, "ad03": 113641,
           "ad04": 124305, "ad05": 178690, "ad06": 63204, "ad07": 57814,
           "ad09": 96351, "ad10": 176260, "ad11": 106606, "ad12": 149395,
           "ad14": 14011, "ad15": 9900, "ad16": 36607, "ad17": 40368}

def find_no_gesture_intervals(gestures, duration):
    """Find time intervals where no gestures occur"""
    # Convert to milliseconds for consistency
    duration_ms = duration * 1000
    
    # Sort gestures by start time
    sorted_gestures = sorted(gestures, key=lambda x: x['start_time'])
    
    # Find gaps between gestures
    gaps = []
    last_end = 0
    
    for gesture in sorted_gestures:
        start = gesture['start_time'] * 1000  # Convert to ms
        end = gesture['end_time'] * 1000
        
        if start > last_end:
            gaps.append({
                'start_time': last_end / 1000,  # Convert back to seconds
                'end_time': start / 1000
            })
        last_end = max(last_end, end)
    
    # Add final gap if there is one
    if last_end < duration_ms:
        gaps.append({
            'start_time': last_end / 1000,
            'end_time': duration
        })
    
    return gaps

def find_matching_no_gesture_clip(gaps, gesture_duration):
    """Find a random gap that can accommodate the gesture duration"""
    valid_gaps = [gap for gap in gaps 
                 if (gap['end_time'] - gap['start_time']) >= gesture_duration]
    
    if not valid_gaps:
        return None
        
    gap = random.choice(valid_gaps)
    max_start = gap['end_time'] - gesture_duration
    
    # Pick a random start time within the gap
    random_start = random.uniform(gap['start_time'], max_start)
    return {
        'start_time': random_start,
        'end_time': random_start + gesture_duration
    }

def parse_gesture_file(file_path, offset):
    gestures = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split('\t') if p.strip()]
            try:
                if len(parts) >= 3:
                    gesture_type = parts[0]
                    start_time = int(float(parts[1])) + offset
                    end_time = int(float(parts[2])) + offset
                    
                    if end_time > start_time and start_time >= 0:
                        gestures.append({
                            'type': gesture_type,
                            'start_time': start_time / 1000.0,
                            'end_time': end_time / 1000.0
                        })
            except (ValueError, IndexError) as e:
                continue
    return gestures

def extract_clip(input_file, output_file, start_time, end_time):
    try:
        duration = end_time - start_time
        cmd = [
            'ffmpeg', '-y',
            '-ss', str(start_time),
            '-i', input_file,
            '-t', str(duration),
            '-c:v', 'libx264',
            '-c:a', 'aac',
            output_file
        ]
        subprocess.run(cmd, check=True, capture_output=True)
        return os.path.exists(output_file)
    except subprocess.CalledProcessError as e:
        print(f"Error extracting clip: {e}")
        return False

def process_video(txt_file, output_dir):
    base_name = os.path.splitext(os.path.basename(txt_file))[0].replace('_final', '')
    video_id = base_name[:4]
    
    if video_id not in offsets:
        print(f"No offset found for video {video_id}")
        return
        
    video_file = os.path.join(os.path.dirname(txt_file), f"{base_name}_speakerview480480.mp4")
    
    if not os.path.exists(video_file):
        print(f"No matching video file found for {txt_file}")
        return
    
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'gestures'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'no_gestures'), exist_ok=True)
    
    with VideoFileClip(video_file) as video:
        duration = video.duration
    
    gestures = parse_gesture_file(txt_file, offsets[video_id])
    print(f"Found {len(gestures)} gestures in {txt_file}")
    
    # Find gaps where no gestures occur
    gaps = find_no_gesture_intervals(gestures, duration)
    
    for idx, gesture in enumerate(gestures):
        gesture_duration = gesture['end_time'] - gesture['start_time']
        
        if gesture['end_time'] <= duration:
            # Process gesture clip
            safe_type = "".join(c if c.isalnum() else "_" for c in gesture['type'])
            gesture_output = os.path.join(output_dir, 'gestures', 
                                        f"ECOLANG_{base_name}_{idx:04d}_{safe_type}.mp4")
            
            if not os.path.exists(gesture_output):
                print(f"Extracting gesture {gesture['type']} from {base_name}: "
                      f"{gesture['start_time']:.2f}s - {gesture['end_time']:.2f}s")
                
                if extract_clip(video_file, gesture_output, 
                              gesture['start_time'], gesture['end_time']):
                    print(f"Successfully extracted gesture: {gesture_output}")
                    
                    # Find and extract matching no-gesture clip
                    no_gesture = find_matching_no_gesture_clip(gaps, gesture_duration)
                    if no_gesture:
                        no_gesture_output = os.path.join(output_dir, 'no_gestures',
                                                       f"ECOLANG_{base_name}_{idx:04d}_no_gesture.mp4")
                        
                        print(f"Extracting matching no-gesture clip: "
                              f"{no_gesture['start_time']:.2f}s - {no_gesture['end_time']:.2f}s")
                        
                        if extract_clip(video_file, no_gesture_output,
                                      no_gesture['start_time'], no_gesture['end_time']):
                            print(f"Successfully extracted no-gesture clip: {no_gesture_output}")
                        else:
                            print(f"Failed to extract no-gesture clip")
                    else:
                        print(f"Could not find suitable no-gesture interval of duration {gesture_duration:.2f}s")
                else:
                    print(f"Failed to extract gesture clip")

def main():
    input_dir = "trainingraw"
    output_dir = "traininggestures"
    
    if not os.path.exists(input_dir):
        print(f"Input directory {input_dir} does not exist!")
        return
        
    txt_files = glob.glob(f"{input_dir}/*final.txt")
    if not txt_files:
        print(f"No .txt files found in {input_dir}")
        return
        
    print(f"Found {len(txt_files)} text files to process")
    for txt_file in txt_files:
        print(f"\nProcessing: {txt_file}")
        process_video(txt_file, output_dir)

if __name__ == "__main__":
    main()

Found 16 text files to process

Processing: trainingraw\ad00_final.txt
Found 195 gestures in trainingraw\ad00_final.txt
Extracting gesture ObjMan from ad00: 467.07s - 477.80s
Successfully extracted gesture: traininggestures\gestures\ECOLANG_ad00_0000_ObjMan.mp4
Extracting matching no-gesture clip: 2184.23s - 2194.95s
Successfully extracted no-gesture clip: traininggestures\no_gestures\ECOLANG_ad00_0000_no_gesture.mp4
Extracting gesture ObjMan from ad00: 479.11s - 480.31s
Successfully extracted gesture: traininggestures\gestures\ECOLANG_ad00_0001_ObjMan.mp4
Extracting matching no-gesture clip: 1555.39s - 1556.59s
Successfully extracted no-gesture clip: traininggestures\no_gestures\ECOLANG_ad00_0001_no_gesture.mp4
Extracting gesture ObjMan from ad00: 480.83s - 488.18s
Successfully extracted gesture: traininggestures\gestures\ECOLANG_ad00_0002_ObjMan.mp4
Extracting matching no-gesture clip: 1857.74s - 1865.09s
Successfully extracted no-gesture clip: traininggestures\no_gestures\ECOLANG_ad